In [ ]:
# /data/chocula/spa-u-cdmssoft/data/LEDdata/23231213_192731

# 23231213_192731_F0012.mid.gz



In [ ]:
import rawio
import numpy as np
from scipy.signal import butter, lfilter
import matplotlib.pyplot as plt
import cdms
print("CDMS Software Version", cdms.get_global_version()) # print out the CDMS software version

In [ ]:
def butter_lowpass_filter(data, cutoff_frequency, sampling_rate=625000, order=4): # cutoff_frequency in kHz
    cutoff_frequency *= 1000
    nyquist = 0.5 * sampling_rate
    normal_cutoff = cutoff_frequency / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = lfilter(b, a, data)
    return y

In [ ]:
#chanNames=["PAS1","PBS1","PCS1","PDS1","PES1","PFS1","PAS2","PBS2","PCS2","PDS2","PES2","PFS2"]
chanNames=["PBS1","PCS1","PDS1","PES1","PFS1","PAS2","PBS2","PCS2","PDS2","PES2","PFS2"] # T3Z3 doesn't have PAS1

In [ ]:
for i in chanNames:
    print(f"Now looking at {i}")

In [ ]:
import os
import glob

print(os.path.exists("/home/yanliusp/23231213_192731"))
print(os.path.isdir("/home/yanliusp/23231213_192731"))
print(glob.glob("/home/yanliusp/23231213_192731*"))
print(glob.glob("/home/yanliusp/*.mid.gz"))

In [ ]:
import glob
import pickle
import os

DATA_DIR = "/projects/standard/yanliusp/shared/data/CDMS/CUTE/R37/Raw/23231213_192731"
PICKLE_EVENTS = "all_events.pkl"

if os.path.exists(PICKLE_EVENTS):
    with open(PICKLE_EVENTS, "rb") as f:
        all_events = pickle.load(f)
    print(f"Loaded {len(all_events)} events from pickle cache: {PICKLE_EVENTS}")
else:
    all_files = sorted(glob.glob(f"{DATA_DIR}/*.mid.gz"))
    print(f"find {len(all_files)} files")

    all_events = []
    for filepath in all_files:
        myreader = rawio.RawDataReader(filepath)
        nb = myreader.get_nb_events()
        evs = myreader.read_events(
            output_format  = 2,
            skip_empty     = True,
            nb_events      = nb["NbEventsNotEmpty"],
            trigger_types  = [1],
            detector_nums  = [1, 2, 3],
            channel_names  = chanNames
        )
        all_events.extend(evs)
        print(f"{filepath.split('/')[-1]}: {len(evs)} events")

    with open(PICKLE_EVENTS, "wb") as f:
        pickle.dump(all_events, f)
    print(f"Saved {len(all_events)} events to pickle cache: {PICKLE_EVENTS}")

print(f"\nto: {len(all_events)} events")

In [ ]:
events = all_events
print(f"Using all_events: {len(events)} events from 12 files")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CLEAN PIPELINE
# Step 1  Helper functions
# Step 2  Template calculation (filter -> align -> average)
# Step 3  Glitch event dictionary
# Step 4  Exponential fitting of template decay tail
# ═══════════════════════════════════════════════════════════════════════════════

from scipy.optimize import curve_fit

def find_rise(y_norm, x0_ms, gap=0.05, x_search_end=29.0, rise_threshold=0.2):
    """Scan from 0 ms for the first point where signal rises > rise_threshold
    within gap ms. Returns (rise_ms, rise_idx) or (None, None)."""
    x_target = 0.0
    while x_target < x_search_end:
        idx     = np.argmin(np.abs(x0_ms - x_target))
        idx_new = np.argmin(np.abs(x0_ms - (x_target + gap)))
        if y_norm[idx_new] - y_norm[idx] > rise_threshold:
            return x_target, idx
        x_target += gap
    return None, None

def preprocess_event(y_raw, baseline_n=5000, snr_threshold=5.0, baseline_threshold=33100,
                     fs=625000, glitch_safe_ms=24.0, post_pulse_ms=35.0):
    """Quality cuts using SNR-based threshold.
    Selects quiet region based on rise position:
      rise late  -> front of trace (before glitch window at ~24.9 ms)
      rise early -> tail of trace (after pulse decay)
    Returns (y_normalised, noise_std_norm, True) or (None, 0.0, False).
    noise_std_norm = noise_std / peak  (noise level in normalised units)."""
    y = y_raw.astype(np.float64)
    baseline = np.mean(y[:baseline_n])
    if abs(baseline) >= baseline_threshold:
        return None, 0.0, False
    y    = y - baseline
    peak = np.max(y)
    if peak <= 0:
        return None, 0.0, False

    # rough normalisation just for rise detection
    y_rough = y / peak
    n       = len(y)
    x0_ms   = np.arange(n) / fs * 1000

    _, rise_idx = find_rise(y_rough, x0_ms)

    glitch_safe_idx = int(glitch_safe_ms * fs / 1000)  # last safe sample before glitch window
    post_pulse_idx  = int(post_pulse_ms  * fs / 1000)  # first safe sample after pulse decay

    if rise_idx is None:
        quiet = y[:baseline_n]
    elif rise_idx > glitch_safe_idx:
        # rise is late -> use front region, safely before glitch window
        quiet = y[:glitch_safe_idx]
    else:
        # rise is early -> use tail region, after pulse has decayed
        quiet = y[post_pulse_idx:] if post_pulse_idx < n else y[-baseline_n:]

    if len(quiet) < 100:
        return None, 0.0, False

    noise_std = np.std(quiet)
    if noise_std <= 0 or peak / noise_std <= snr_threshold:
        return None, 0.0, False

    return y / peak, noise_std / peak, True

def align_event(y_norm, x0_ms, align_at_ms=25.3, gap=0.05, x_search_end=29.0, rise_threshold=0.2):
    """Shift time axis so rising edge lands at align_at_ms. Returns x_aligned or None."""
    rise_ms, _ = find_rise(y_norm, x0_ms, gap, x_search_end, rise_threshold)
    if rise_ms is None:
        return None
    return x0_ms - (rise_ms - align_at_ms)

def roll_to_align(y_norm, x0_ms, align_at_ms=25.3, gap=0.05, x_search_end=29.0, rise_threshold=0.2):
    """Roll y so rise lands at align_at_ms -- puts all traces on same sample grid for averaging."""
    rise_ms, rise_idx = find_rise(y_norm, x0_ms, gap, x_search_end, rise_threshold)
    if rise_ms is None:
        return None
    ref_idx = int(np.argmin(np.abs(x0_ms - align_at_ms)))
    return np.roll(y_norm, ref_idx - rise_idx)

def is_glitch(x_aligned, y_norm, noise_std_norm, pre_window=(24.9, 25.25), glitch_sigma=5.0):
    """True if pre-pulse dip < -glitch_sigma x noise_std_norm."""
    mask = (x_aligned >= pre_window[0]) & (x_aligned <= pre_window[1])
    if not np.any(mask):
        return False
    return float(np.min(y_norm[mask])) < -glitch_sigma * noise_std_norm

print("Helper functions ready.")

In [ ]:
import pickle
import os
from scipy.optimize import curve_fit

# ── Step 2 & 3: Template + glitch/good event dictionaries ─────────────────────
PICKLE_TMPL   = "template_dict.pkl"
PICKLE_GLITCH = "glitch_event_dict.pkl"
PICKLE_GOOD   = "good_event_dict.pkl"

# Set True to force re-running after algorithm changes
FORCE_RERUN = True

fs  = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']

if not FORCE_RERUN and (os.path.exists(PICKLE_TMPL) and os.path.exists(PICKLE_GLITCH)
        and os.path.exists(PICKLE_GOOD)):
    with open(PICKLE_TMPL,   "rb") as f: template_dict     = pickle.load(f)
    with open(PICKLE_GLITCH, "rb") as f: glitch_event_dict = pickle.load(f)
    with open(PICKLE_GOOD,   "rb") as f: good_event_dict   = pickle.load(f)
    print("Loaded from pickle cache.")
    for Z_choice in ['Z1', 'Z2', 'Z3']:
        for i_det in lst:
            g = glitch_event_dict[Z_choice][i_det]
            print(f"{Z_choice} {i_det:6s}: {len(good_event_dict[Z_choice][i_det]):3d} good,  glitches: {str(g) if g else 'none'}")
else:
    template_dict     = {}
    good_event_dict   = {}
    glitch_event_dict = {}

    for Z_choice in ['Z1', 'Z2', 'Z3']:
        template_dict[Z_choice]     = {}
        good_event_dict[Z_choice]   = {}
        glitch_event_dict[Z_choice] = {}

        for i_det in lst:
            x0_ms = np.arange(len(events[0][Z_choice][i_det])) / fs * 1000

            aligned_traces = []
            good_events    = []
            glitch_events  = []

            for i in range(len(events)):
                y_raw = events[i][Z_choice][i_det].astype(np.int32)

                y_norm, noise_std_norm, ok = preprocess_event(y_raw)
                if not ok:
                    continue

                x_aligned = align_event(y_norm, x0_ms)
                if x_aligned is None:
                    continue

                if is_glitch(x_aligned, y_norm, noise_std_norm):
                    glitch_events.append(i)
                    continue

                y_rolled = roll_to_align(y_norm, x0_ms)
                if y_rolled is None:
                    continue

                aligned_traces.append(y_rolled)
                good_events.append(i)

            good_event_dict[Z_choice][i_det]   = good_events
            glitch_event_dict[Z_choice][i_det] = glitch_events
            template_dict[Z_choice][i_det]     = np.mean(aligned_traces, axis=0) if aligned_traces else None

            g_str = str(glitch_events) if glitch_events else "none"
            print(f"{Z_choice} {i_det:6s}: {len(good_events):3d} good events,  glitches: {g_str}")

    with open(PICKLE_TMPL,   "wb") as f: pickle.dump(template_dict,     f)
    with open(PICKLE_GLITCH, "wb") as f: pickle.dump(glitch_event_dict, f)
    with open(PICKLE_GOOD,   "wb") as f: pickle.dump(good_event_dict,   f)
    print("\nSaved pickle caches: template_dict.pkl, glitch_event_dict.pkl, good_event_dict.pkl")

In [ ]:
# ── Step 2 (cont.): Plot templates — 3×11 grid ───────────────────────────────
fs     = 625000
lst    = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
Z_list = ['Z1', 'Z2', 'Z3']

n_rows, n_cols = len(Z_list), len(lst)   # 3 × 11
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 2.4, n_rows * 2.6),
                          sharex=True, sharey=False)

for r, Z_choice in enumerate(Z_list):
    for c, i_det in enumerate(lst):
        ax   = axes[r, c]
        tmpl = template_dict[Z_choice][i_det]
        n    = len(good_event_dict[Z_choice][i_det])

        if tmpl is not None:
            t_ms = np.arange(len(tmpl)) / fs * 1000
            ax.plot(t_ms, tmpl, lw=1, color='steelblue')
            ax.set_xlim(24.5, 32)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=8)
            ax.set_facecolor('#f5f5f5')

        ax.set_title(f"{i_det}  N={n}", fontsize=7, pad=2)
        ax.tick_params(labelsize=6)

        if c == 0:
            ax.set_ylabel(Z_choice, fontsize=9, labelpad=3)
        else:
            ax.set_yticklabels([])

        if r < n_rows - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel('ms', fontsize=7)

fig.suptitle('Templates — Z1 / Z2 / Z3  ×  all channels', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 4: Full-pulse fitting ─────────────────────────────────────────────────
from scipy.optimize import curve_fit
import warnings

samplerate = 625000

def two_exp_fit(x, amp1, t1, t2, baseline, pretrigger):
    return np.where(
        x <= pretrigger, baseline,
        -(amp1 * np.exp(-(x - pretrigger) / t1 / samplerate)
          - amp1 * np.exp(-(x - pretrigger) / t2 / samplerate)) + baseline
    )

def three_exp_fit(x, amp1, amp2, t1, t2, t3, baseline, pretrigger):
    return np.where(
        x <= pretrigger, baseline,
        -((amp1 + amp2) * np.exp(-(x - pretrigger) / t1 / samplerate)
          - amp1         * np.exp(-(x - pretrigger) / t2 / samplerate)
          - amp2         * np.exp(-(x - pretrigger) / t3 / samplerate)) + baseline
    )

def auto_guess(tmpl, win_start, samplerate=625000):
    peak_idx = int(np.argmax(tmpl))
    above5 = np.where(tmpl[win_start:peak_idx] >= 0.05)[0]
    pretrigger_guess = float(win_start + above5[0]) - 50 if len(above5) > 0 else float(peak_idx) - 200
    pretrigger_guess = max(float(win_start), pretrigger_guess)
    post = tmpl[peak_idx:]
    inv_e_arr = np.where(post <= 1.0 / np.e)[0]
    t2_guess = inv_e_arr[0] / samplerate if len(inv_e_arr) > 0 else 5e-3
    return pretrigger_guess, t2_guess * 0.05, t2_guess, t2_guess * 5.0


lst      = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
Z_list   = ['Z1', 'Z2', 'Z3']
WIN_START = 14000

fit_params_dict = {}
_plot = {}   # (t_ms, y, y_fit, fit_tag, t2_ms, n)

for Z_choice in Z_list:
    fit_params_dict[Z_choice] = {}
    _plot[Z_choice] = {}
    for i_det in lst:
        raw_tmpl = template_dict[Z_choice][i_det]
        if raw_tmpl is None:
            fit_params_dict[Z_choice][i_det] = None
            _plot[Z_choice][i_det] = None
            continue

        tmpl = raw_tmpl / np.max(raw_tmpl)
        x    = np.arange(WIN_START, len(tmpl), dtype=float)
        y    = tmpl[WIN_START:]
        pt_g, t1_g, t2_g, t3_g = auto_guess(tmpl, WIN_START)

        fit_ok = False
        for try3 in [True, False]:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    if try3:
                        popt, _ = curve_fit(
                            three_exp_fit, x, y,
                            p0=[0.5, 0.5, t1_g, t2_g, t3_g, 0.0, pt_g], maxfev=int(1e5))
                        amp1, amp2, t1, t2, t3, bl, pt = popt
                        y_fit   = three_exp_fit(x, *popt)
                        fit_tag = '3-exp'
                        label   = f't1={t1*1e3:.3f}  t2={t2*1e3:.2f}  t3={t3*1e3:.2f} ms'
                    else:
                        popt, _ = curve_fit(
                            two_exp_fit, x, y,
                            p0=[1.0, t1_g, t2_g, 0.0, pt_g], maxfev=int(1e5))
                        amp1, t1, t2, bl, pt = popt
                        y_fit   = two_exp_fit(x, *popt)
                        fit_tag = '2-exp'
                        label   = f't1={t1*1e3:.3f}  t2={t2*1e3:.2f} ms'
                fit_params_dict[Z_choice][i_det] = (fit_tag, popt)
                t_ms   = x / samplerate * 1000
                n      = len(good_event_dict[Z_choice][i_det])
                _plot[Z_choice][i_det] = (t_ms, y, y_fit, fit_tag, t2 * 1e3, label, n)
                fit_ok = True
                print(f'  {Z_choice} {i_det}: [{fit_tag}] {label}')
                break
            except RuntimeError:
                if try3:
                    print(f'  {Z_choice} {i_det}: 3-exp failed, trying 2-exp')
                else:
                    print(f'  {Z_choice} {i_det}: both fits failed')
                    fit_params_dict[Z_choice][i_det] = None
                    _plot[Z_choice][i_det] = None

# ── Grid plot 3 × 11 ──────────────────────────────────────────────────────────
n_rows, n_cols = len(Z_list), len(lst)
fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 2.4, n_rows * 2.6),
                          sharex=False, sharey=False)

for r, Z_choice in enumerate(Z_list):
    for c, i_det in enumerate(lst):
        ax   = axes[r, c]
        data = _plot[Z_choice][i_det]

        if data is not None:
            t_ms, y, y_fit, fit_tag, t2_ms, label, n = data
            ax.plot(t_ms, y,     color='steelblue', lw=1,   label=f'N={n}')
            ax.plot(t_ms, y_fit, color='red',       lw=1.5, ls='--')
            ax.set_xlim(24.5, 36)
            ax.text(0.97, 0.97, label, ha='right', va='top',
                    transform=ax.transAxes, fontsize=4.5, color='darkred')
            ttl = f'{i_det} [{fit_tag}]\nt2={t2_ms:.2f} ms'
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=8)
            ax.set_facecolor('#f5f5f5')
            ttl = i_det

        ax.set_title(ttl, fontsize=6.5, pad=2, linespacing=1.3)
        ax.tick_params(labelsize=6)
        if c == 0:
            ax.set_ylabel(Z_choice, fontsize=9, labelpad=3)
        else:
            ax.set_yticklabels([])
        if r < n_rows - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel('ms', fontsize=7)

fig.suptitle('Template fits — Z1 / Z2 / Z3  ×  all channels  (blue = template, red = fit)',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Summary: fitted time constants table + bar chart ─────────────────────────
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']

print(f"{'':14s} {'t1-rise [ms]':>14s} {'t2-decay [ms]':>14s} {'t3-decay [ms]':>14s}")
for Z_choice in ['Z1', 'Z2', 'Z3']:
    for i_det in lst:
        entry = fit_params_dict[Z_choice][i_det]
        if entry is None:
            print(f'  {Z_choice} {i_det:<6s}:  N/A')
        elif entry[0] == '3-exp':
            amp1, amp2, t1, t2, t3, baseline, pt = entry[1]
            print(f'  {Z_choice} {i_det:<6s}: {t1*1e3:>12.3f}  {t2*1e3:>12.2f}  {t3*1e3:>12.2f}')
        else:
            amp1, t1, t2, baseline, pt = entry[1]
            print(f'  {Z_choice} {i_det:<6s}: {t1*1e3:>12.3f}  {t2*1e3:>12.2f}  {"N/A":>12s}  (2-exp)')

# bar chart — t2 (primary decay) per channel
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, Z_choice in zip(axes, ['Z1', 'Z2', 'Z3']):
    t2_vals, colors = [], []
    for ch in lst:
        entry = fit_params_dict[Z_choice].get(ch)
        if entry is None:
            t2_vals.append(0); colors.append('lightgray')
        elif entry[0] == '3-exp':
            t2_vals.append(entry[1][3] * 1e3); colors.append('steelblue')
        else:
            t2_vals.append(entry[1][2] * 1e3); colors.append('orange')
    bars = ax.bar(lst, t2_vals, color=colors)
    ax.set_title(Z_choice)
    ax.set_xlabel('Channel')
    ax.tick_params(axis='x', rotation=45)
    for bar, v in zip(bars, t2_vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('t2-decay [ms]')
fig.suptitle('Phonon decay time constants (t2) by channel', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Glitch event summary: one figure per event, 3 detectors × 11 channels ─────
#
# Red border  = this (Z, channel) flagged the event as a glitch
# Red shading = pre-pulse glitch window [24.9, 25.25] ms
# Blue   = aligned pulse
# Orange = pulse present but alignment failed (shown in original time)
# Grey   = no significant pulse (noise / flat — still plotted)

fs    = 625000
lst   = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
Z_list = ['Z1', 'Z2', 'Z3']

# Collect all unique glitch event indices across every detector/channel
all_glitch_events = set()
for Z in Z_list:
    for ch in lst:
        all_glitch_events.update(glitch_event_dict[Z][ch])
all_glitch_events = sorted(all_glitch_events)
print(f"Unique glitch events to visualize: {all_glitch_events}")

for ev_idx in all_glitch_events:
    if ev_idx >= len(events):
        print(f"Event {ev_idx} out of range, skipping.")
        continue

    n_rows, n_cols = len(Z_list), len(lst)   # 3 × 11
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 1.9, n_rows * 2.2),
        sharey=False
    )
    fig.suptitle(f"Glitch Event {ev_idx}  —  All Detectors × All Channels",
                 fontsize=11, y=1.01)

    for r_idx, Z_choice in enumerate(Z_list):
        x0_ms = np.arange(len(events[0][Z_choice][lst[0]])) / fs * 1000

        for c_idx, i_det in enumerate(lst):
            ax = axes[r_idx, c_idx]

            y_raw  = events[ev_idx][Z_choice][i_det].astype(np.int32)
            y_norm, noise_std_norm, ok = preprocess_event(y_raw)

            if ok:
                x_aligned = align_event(y_norm, x0_ms)
                if x_aligned is not None:
                    # aligned pulse — zoom to pulse region
                    mask = (x_aligned >= 24.3) & (x_aligned <= 27.5)
                    ax.plot(x_aligned[mask], y_norm[mask], lw=0.8, color='steelblue')
                    ax.axvspan(24.9, 25.25, alpha=0.18, color='red', zorder=0)
                    ax.axhline(0, color='k', lw=0.4, ls='--')
                else:
                    # pulse present but alignment failed — zoom around peak in original time
                    peak_ms = x0_ms[int(np.argmax(y_norm))]
                    lo = max(0.0, peak_ms - 1.5)
                    hi = peak_ms + 3.5
                    mask = (x0_ms >= lo) & (x0_ms <= hi)
                    ax.plot(x0_ms[mask], y_norm[mask], lw=0.8, color='orange')
                    ax.axhline(0, color='k', lw=0.4, ls='--')
                    ax.text(0.03, 0.95, 'no align', ha='left', va='top',
                            transform=ax.transAxes, fontsize=5, color='darkorange')
            else:
                # no significant pulse — plot baseline-subtracted signal normalised to max-abs
                y = y_raw.astype(np.float64)
                y = y - np.mean(y[:5000])
                amp = np.max(np.abs(y))
                y_plot = y / amp if amp > 0 else y
                mask = (x0_ms >= 24.3) & (x0_ms <= 27.5)
                ax.plot(x0_ms[mask], y_plot[mask], lw=0.8, color='#888888')
                ax.axhline(0, color='k', lw=0.4, ls='--')
                ax.text(0.03, 0.95, 'no pulse', ha='left', va='top',
                        transform=ax.transAxes, fontsize=5, color='gray')

            # Red border if this (Z, ch) flagged this event as a glitch
            is_flagged = ev_idx in glitch_event_dict[Z_choice][i_det]
            border_color = 'red' if is_flagged else '#bbbbbb'
            border_width = 2.0  if is_flagged else 0.5
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(border_width)

            ax.tick_params(labelsize=5, pad=1)
            if r_idx == 0:
                ax.set_title(i_det, fontsize=7, pad=3)
            if c_idx == 0:
                ax.set_ylabel(Z_choice, fontsize=8, labelpad=2)
            else:
                ax.set_yticklabels([])
            if r_idx < n_rows - 1:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel('ms', fontsize=5)

    plt.tight_layout()
    plt.show()
    print(f"Event {ev_idx} plotted.")

# Do the following for just one Channel for 31 events for 3 detectors
## 1. remove events without pulses - 
## 2. align pulses - 
    ### 2.1 find the time difference/offset - max and min
    ### 2.2 normalization 
        #### - calculate baseline, subtract baseline so it's zero
        #### - scale max y-axis valve to the same number (1)
## 3. calculate average
## template


# iterate over files.

In [ ]:
DATA_DIR = "/projects/standard/yanliusp/shared/data/CDMS/CUTE/R37/Raw/23231213_192731"

# F0003: 400 events (main analysis file)
# F0012: 31 events
filepath_F0003 = f"{DATA_DIR}/23231213_192731_F0004.mid.gz"
filepath_F0012 = f"{DATA_DIR}/23231213_192731_F0012.mid.gz"

filepath = filepath_F0003   # change to filepath_F0012 to use the other file

myreader = rawio.RawDataReader(filepath)

odb     = myreader.get_full_odb(odb_dir='/Detectors/Det03/Settings/Phonon')
runinfo = myreader.get_run_info()
hvinfo  = myreader.get_hv_info()

In [ ]:
events = myreader.read_events(
    output_format=2,
    skip_empty=True,
    nb_events=400,
    trigger_types=[1],
    detector_nums=[1,2,3],
    channel_names=chanNames
)

In [ ]:
nb_events = myreader.get_nb_events()

In [ ]:
nb_events["NbEventsNotEmpty"]
nb_events

In [ ]:
nb_events

In [ ]:
events = myreader.read_events(output_format=2,
                                  skip_empty=True,
                                  #event_nums=[50019,50306],
                                  nb_events=nb_events["NbEventsNotEmpty"],
                                  trigger_types=[1],
                                  detector_nums=[1,2,3],
                                  channel_names=chanNames)

In [ ]:
type(events)

In [ ]:
len(events)

In [ ]:
events[0].keys()

In [ ]:
events[0]["Z3"]

In [ ]:
fs = 625000

x = range(len(events[0]["Z3"]["PFS1"]))
y = events[0]["Z3"]["PFS1"]
x = np.array(x)*1/fs*1000
#plt.plot(x, y, label = "PSF1")



for i in chanNames:
    y = events[0]["Z3"][i]
    plt.plot(x, y, label = i)
    

plt.legend()
plt.xlabel("time bin[ms]")
plt.ylabel("ADC units")
plt.title("event0")
#plt.xlim(0, 10)


plt.show()

In [ ]:
fs = 625000

x = range(len(events[0]["Z3"]["PFS1"]))

x = np.array(x)*1/fs*1000
#plt.plot(x, y, label = "PSF1")



for i in range(len(events)):
    y = events[i]["Z3"]["PBS1"]
    plt.plot(x, y, label = i)
    

plt.legend()
plt.xlabel("time bin[ms]")
plt.ylabel("ADC units")
plt.title("PBS1")
plt.xlim(25, 30)


plt.show()

In [ ]:
#在上面那个样板的代码下进行循环Z1，Z2，Z3和各个通道，初始所有的循环
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
#for Z_choice in ['Z1', 'Z2','Z3']:
for Z_choice in ['Z3']:
    for i_det in lst:
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        i = 12

        
        y = (events[i][Z_choice][i_det]).astype(np.int32)
        baseline = np.mean(y[30000:])#can be changed based on the time event occurs
        y = y - baseline
        y2 = butter_lowpass_filter(y,10)
        #y = y/max(y)
                 
                
               
        gap=0.05
        x_target = 0
        found = False
                
                
                   
                
        plt.plot(x0, y, label=f"event: {i}")
        plt.plot(x0, y2, label = f"event: {i}")
            
        
        plt.legend()
        plt.xlabel("time bin[ms]")
        plt.ylabel("ADC units")
        plt.title(f"{Z_choice} {i_det}")
        plt.xlim(0, 10)
        
        
        plt.show()

In [ ]:
#在上面那个样板的代码下进行循环Z1，Z2，Z3和各个通道，初始所有的循环
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
#for Z_choice in ['Z1', 'Z2','Z3']:
for Z_choice in ['Z3']:

    for i_det in lst:
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        i = 18

        
        y = (events[i][Z_choice][i_det]).astype(np.int32)
        baseline = np.mean(y[:5000])#can be changed based on the time event occurs
        y = y - baseline
        #y = y/max(y)
                 
                
               
        gap=0.05
        x_target = 0
        found = False
                
                
                   
                
        #plt.scatter(x, y, label=f"event: {i}")
        plt.plot(x, y, label = f"event: {i}")
            
        
        plt.legend()
        plt.xlabel("time bin[ms]")
        plt.ylabel("ADC units")
        plt.title(f"{Z_choice} {i_det}")
        #plt.xlim(25.3, 25.38)
        
        
        plt.show()

In [ ]:
#在上面那个样板的代码下进行循环Z1，Z2，Z3和各个通道，初始所有的循环
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
for Z_choice in ['Z1', 'Z2','Z3']:
    for i_det in lst:
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        
        for i in range(len(events)):
            y = (events[i][Z_choice][i_det]).astype(np.int32)
            if max(y)-min(y)>2000: #
                y = y - np.mean(y[:5000])
                y = y/max(y)
                 
                
               
                gap=0.05
                x_target = 25.67
                found = False
                while x_target < 26:
                
                    idx = np.argmin(np.abs(x0 - x_target))
                    idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
                    if y[idx_new]- y[idx] > 0.2:
                        x = x0-(x_target-25.3)
                        found = True
                        break
                    x_target += gap
                if found:
                    plt.plot(x, y, label = f"event: {i}")
            
        
        plt.legend()
        plt.xlabel("time bin[ms]")
        plt.ylabel("ADC units")
        plt.title(f"{Z_choice} {i_det}")
        plt.xlim(25, 27)
        
        
        plt.show()

In [ ]:
#select glitch event loop every detectors and channels then plot
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']

for Z_choice in ['Z1', 'Z2','Z3']:
    for i_det in lst:
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        
        for i in range(len(events)):
            y = (events[i][Z_choice][i_det]).astype(np.int32)
            baseline = np.mean(y[:5000])
            if max(y)-min(y)>2000 and abs(baseline)<33100: #event12在PBS1，Z3里差不多33200，这样把event12过滤
                y = y - baseline
                y = y/max(y)
                 
                
               
                gap=0.05
                x_target = 0
                found = False
                while x_target < 29:
                
                    idx = np.argmin(np.abs(x0 - x_target))
                    idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
                    if y[idx_new]- y[idx] > 0.2:
                        x = x0-(x_target-25.3)
                        found = True
                        break
                    x_target += gap
                if found:
                    # glitch window: pulse前一点
                    mask = (x >= 24.9) & (x <= 25.25)
                    pre_min = np.min(y[mask])

                    if pre_min < -0.05:   # 这个阈值可调
                        
                        #plt.plot(x, y, label=f"event: {i}")
                        plt.plot(x, y, label = f"event: {i}")
            
        
        plt.legend()
        plt.xlabel("time bin[ms]")
        plt.ylabel("ADC units")
        plt.title(f"{Z_choice} {i_det}")
        plt.xlim(22, 30)
        
        
        plt.show()


In [ ]:
#to get a dictionary of glitch event for all detectors (nested dictionary for chennals and the list for events)
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
glitch_event_dic= {}
for Z_choice in ['Z1', 'Z2','Z3']:
    glitch_event_dic[Z_choice] = {}
    for i_det in lst:
        glitch_event_dic[Z_choice][i_det] = []
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        
        for i in range(len(events)):
            y = (events[i][Z_choice][i_det]).astype(np.int32)
            baseline = np.mean(y[:5000])
            if max(y)-min(y)>2000 and abs(baseline)<33100: #event12在PBS1，Z3里差不多33200，这样把event12过滤
                y = y - baseline
                y = y/max(y)
                 
                
               
                gap=0.05
                x_target = 0
                found = False
                while x_target < 29:
                
                    idx = np.argmin(np.abs(x0 - x_target))
                    idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
                    if y[idx_new]- y[idx] > 0.2:
                        x = x0-(x_target-25.3)
                        found = True
                        break
                    x_target += gap
                if found:
                    # glitch window: pulse前一点
                    mask = (x >= 24.9) & (x <= 25.25)
                    pre_min = np.min(y[mask])

                    if pre_min < -0.05:   # 这个阈值可调
                        #dic[Z_choice,i_det,i] = i
                        glitch_event_dic[Z_choice][i_det].append(i)
                
                        #plt.plot(x, y, label=f"event: {i}")
                        #plt.plot(x, y, label = f"event: {i}")
            
        
        #plt.legend()
        #plt.xlabel("time bin[ms]")
        #plt.ylabel("ADC units")
        #plt.title(f"{Z_choice} {i_det}")
        #plt.xlim(22, 30)
        
        
        #plt.show()

import numpy as np
import os

filename = "glitch_event_dic.npy"

# save dictionary as .npy file
np.save(filename, glitch_event_dic)


print("Current working directory:", os.getcwd())
print("Saved file name:", filename)
print("Full file path:", os.path.abspath(filename))
print(glitch_event_dic)


In [ ]:
import numpy as np, os
# load glitch_event_dic saved by the cell above
glitch_event_dic = np.load("glitch_event_dic.npy", allow_pickle=True).item()
print("Loaded from:", os.path.abspath("glitch_event_dic.npy"))
print(glitch_event_dic)

In [ ]:
#select glitch event loop every detectors and channels then plot
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']

for Z_choice in ['Z1', 'Z2','Z3']:
    for i_det in lst:
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        
        for i in range(len(events)):
            y = (events[i][Z_choice][i_det]).astype(np.int32)
            baseline = np.mean(y[:5000])
            if max(y)-min(y)>2000 and abs(baseline)<33100: #event12在PBS1，Z3里差不多33200，这样把event12过滤
                y = y - baseline
                y = y/max(y)
                 
                
               
                gap=0.05
                x_target = 0
                found = False
                while x_target < 29:
                
                    idx = np.argmin(np.abs(x0 - x_target))
                    idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
                    if y[idx_new]- y[idx] > 0.2:
                        x = x0-(x_target-25.3)
                        found = True
                        break
                    x_target += gap
                if found:
                    # glitch window: pulse前一点
                    mask = (x >= 24.9) & (x <= 25.25)
                    pre_min = np.min(y[mask])

                    if pre_min < -0.05:   # 这个阈值可调
                        
                        #plt.plot(x, y, label=f"event: {i}")
                        plt.plot(x, y, label = f"event: {i}")
            
        
        plt.legend()
        plt.xlabel("time bin[ms]")
        plt.ylabel("ADC units")
        plt.title(f"{Z_choice} {i_det}")
        plt.xlim(22, 30)
        
        
        plt.show()


In [ ]:
for Z_choice in ['Z1', 'Z2','Z3']:
    for i_det in lst:
        fs = 625000
        plotted = False  
        
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        for i in [381]:  
            y = (events[i][Z_choice][i_det]).astype(np.int32)
            baseline = np.mean(y[:5000])
            if max(y)-min(y)>2000 and abs(baseline)<33100:
                y = y - baseline
                y = y/max(y)
                
                gap=0.05
                x_target = 0
                found = False
                while x_target < 29:
                    idx = np.argmin(np.abs(x0 - x_target))
                    idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
                    if y[idx_new]- y[idx] > 0.2:
                        x = x0-(x_target-25.3)
                        found = True
                        break
                    x_target += gap

                if found:
                    mask = (x >= 24.9) & (x <= 25.25)
                    pre_min = np.min(y[mask])
            
                    if pre_min < -0.05:
                        plt.plot(x, y, label = f"event: {i}")
                        plotted = True   
        
        if plotted:  
            plt.legend()
            plt.xlabel("time bin[ms]")
            plt.ylabel("ADC units")
            plt.title(f"{Z_choice} {i_det}")
            plt.xlim(25, 25.5)
            plt.show()
        else:
            plt.close()


In [ ]:
#开始筛选LEDevent
fs = 625000

#x = range(len(events[0]["Z3"]["PFS1"]))

#x = np.array(x)*1/fs*1000
#plt.plot(x, y, label = "PSF1")
x0 = range(len(events[0]["Z3"]["PBS1"]))
x0 = np.array(x0) * 1/fs * 1000


   

for i in range(len(events)):
    y = (events[i]["Z3"]["PBS1"]).astype(np.int32)

    baseline = np.mean(y[:5000])
    if max(y)-min(y)>2000 and abs(baseline)<33000: 
        y = y - baseline
        y = y/max(y)
         
        
       
        gap=0.05
        x_target = 0
        found = False
        while x_target < 29:
    
            idx = np.argmin(np.abs(x0 - x_target))
            idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
            if y[idx_new]- y[idx] > 0.2:
                x = x0-(x_target-25.3)
                found = True
                break
            x_target += gap
    
        if found:
            # glitch window: pulse前一点
            mask = (x >= 24.9) & (x <= 25.25)
            pre_min = np.min(y[mask])

            if pre_min < -0.05:   # 这个阈值可以再调
                
                plt.plot(x, y, label=f"event: {i}")
            #plt.plot(x, y, label = f"event: {i}")
    

plt.legend()
plt.xlabel("time bin[ms]")
plt.ylabel("ADC units")
plt.title("PBS1")
plt.xlim(22, 30)


plt.show()

In [ ]:
#初级成果第一个样板从0开始扫描Z3 PBS1 的筛选代码，先筛选y在对齐x
fs = 625000

#x = range(len(events[0]["Z3"]["PFS1"]))

#x = np.array(x)*1/fs*1000
#plt.plot(x, y, label = "PSF1")
x0 = range(len(events[0]["Z3"]["PBS1"]))
x0 = np.array(x0) * 1/fs * 1000


   

for i in range(len(events)):
    y = (events[i]["Z3"]["PBS1"]).astype(np.int32)

    if max(y)-min(y)>2000: #
        y = y - np.mean(y[31000:])
        y = y/max(y)
         #
        
       
        gap=0.05
        x_target = 0
        found = False
        while x_target < 27:
    
            idx = np.argmin(np.abs(x0 - x_target))
            idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
            if y[idx_new]- y[idx] > 0.2:
                x = x0-(x_target-25.3)
                found = True
                break
            x_target += gap

        if found:
            
            plt.plot(x, y, label = f"event: {i}")
    

plt.legend()
plt.xlabel("time bin[ms]")
plt.ylabel("ADC units")
plt.title("PBS1")
plt.xlim(22, 30)


plt.show()

In [ ]:
#Z3 PBS1 event12 的单独图像
fs = 625000

#x = range(len(events[0]["Z3"]["PFS1"]))

#x = np.array(x)*1/fs*1000
#plt.plot(x, y, label = "PSF1")
x0 = range(len(events[0]["Z3"]["PBS1"]))
x0 = np.array(x0) * 1/fs * 1000



    
i = 9
y = (events[i]["Z3"]["PBS1"]).astype(np.int32)
if max(y)-min(y)>1000: #
    y = y - np.mean(y[:5000])
    y = y/max(y)
    
    
   
    gap=0.05
    x_target = 22
    found = False
    while x_target < 27:
    
        idx = np.argmin(np.abs(x0 - x_target))
        idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
        if y[idx_new]- y[idx] > 0.2:
            x = x0-(x_target-25.3)
            found = True
            break
        x_target += gap
    plt.plot(x, y, label = f"event: {i}")
    

plt.legend()
plt.xlabel("time bin[ms]")
plt.ylabel("ADC units")
plt.title("PBS1")
plt.xlim(0, 30)


plt.show()

In [ ]:
fs = 625000
x0 = range(len(events[0]["Z3"]["PBS1"]))
x0 = np.array(x0) * 1/fs * 1000

i = 12
y = (events[i]["Z3"]["PBS1"]).astype(np.int32)
if max(y)-min(y)>1000:
    y = y - np.mean(y[:5000])
    y = y/max(y)

    gap=0.05
    x_target = 22
    found = False
    while x_target < 27:
        idx = np.argmin(np.abs(x0 - x_target))
        idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
        if y[idx_new]- y[idx] > 0.2:
            x = x0-(x_target-25.3)
            found = True
            break
        x_target += gap
    plt.plot(x, y, label = f"event: {i}")

plt.legend()
plt.xlabel("time bin[ms]")
plt.ylabel("ADC units")
plt.title("PBS1  event12")
plt.xlim(0, 30)
plt.show()

In [ ]:
fs = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
for Z_choice in ['Z1', 'Z2','Z3']:
    for i_det in lst:
        fs = 625000
    
        #x = range(len(events[0]["Z3"]["PFS1"]))
        
        #x = np.array(x)*1/fs*1000
        #plt.plot(x, y, label = "PSF1")
        x0 = range(len(events[0][Z_choice][i_det]))
        x0 = np.array(x0) * 1/fs * 1000
        
        
        for i in range(len(events)):
            y = (events[i][Z_choice][i_det]).astype(np.int32)
            if max(y)-min(y)>2000: #
                y = y - np.mean(y[:5000])
                y = y/max(y)
                 
                
               
                gap=0.05
                x_target = 0
                found = False
                while x_target < 27:
                
                    idx = np.argmin(np.abs(x0 - x_target))
                    idx_new = np.argmin(np.abs(x0 - (x_target+gap)))
                    if y[idx_new]- y[idx] > 0.2:
                        x = x0-(x_target-25.3)
                        found = True
                        break
                    x_target += gap
                if found:
                    plt.plot(x, y, label = f"event: {i}")
            
        
        plt.legend()
        plt.xlabel("time bin[ms]")
        plt.ylabel("ADC units")
        plt.title(f"{Z_choice} {i_det}")
        plt.xlim(22, 30)
        
        
        plt.show()

In [ ]:
i = 12
y = events[i]["Z3"]["PBS1"].astype(np.int32)
y = y - np.mean(y[:5000])
y = y / max(y)

gap = 0.05
x_target = 22

best_rise = -999

while x_target < 27:
    idx = np.argmin(np.abs(x0 - x_target))
    idx_new = np.argmin(np.abs(x0 - (x_target + gap)))
    rise = y[idx_new] - y[idx]
    if rise > best_rise:
        best_rise = rise
    x_target += gap

print(best_rise)

In [ ]:
len(events[0]['Z3']["PBS1"])

In [ ]:
plt.plot(events[0]['Z3']['PBS2'][15500:18000])
plt.plot(events[0]['Z3']['PAS2'][15500:18000])
plt.plot(events[0]['Z3']['PFS2'][15500:18000])
plt.plot(events[0]['Z3']['PES2'][15500:18000])
plt.plot(events[0]['Z3']['PES1'][15500:18000])

In [ ]:
#for chan in chanNames:
#    print(chan)
    #plt.plot(events[1]['Z3'][chan][15800:15900])
plt.plot(events[0]['Z2']["PBS1"][:])

In [ ]:
#for nb in range(len(events)):

plt.plot((events[0]['Z3']["PCS2"][15000+180:17000+180] -32200) * 1/12400)
plt.plot((events[2]['Z3']["PCS2"][15000+25:17000+25] -32200) * 1/12200)
plt.plot((events[4]['Z3']["PCS2"][15000:17000] - 32200) * 1/12300)

In [ ]:
# ── 诊断：SNR 分布 + pre-pulse dip 分布，用于确定筛选倍数 ──────────────────────
fs  = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
N   = 5000

snr_list          = []
glitch_depth_list = []

for Z_choice in ['Z1', 'Z2', 'Z3']:
    for i_det in lst:
        x0_ms = np.arange(len(events[0][Z_choice][i_det])) / fs * 1000
        for i in range(len(events)):
            y_norm, noise_std_norm, ok = preprocess_event(events[i][Z_choice][i_det].astype(np.int32))
            if not ok:
                continue
            rise_ms, rise_idx = find_rise(y_norm, x0_ms)
            if rise_idx is None:
                continue
            noise_std = np.std(y_norm[:N]) if rise_idx > N + 500 else np.std(y_norm[-N:])
            if noise_std <= 0:
                continue
            snr_list.append(1.0 / noise_std)
            x_aligned = align_event(y_norm, x0_ms)
            if x_aligned is None:
                continue
            mask = (x_aligned >= 24.9) & (x_aligned <= 25.25)
            if np.any(mask):
                glitch_depth_list.append(float(np.min(y_norm[mask])) / noise_std)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(snr_list, bins=60, color='steelblue', edgecolor='white')
ax1.set_xlabel('SNR  (peak / noise_std)')
ax1.set_ylabel('count')
ax1.set_title('Good event SNR distribution')

ax2.hist(glitch_depth_list, bins=60, color='steelblue', edgecolor='white')
ax2.axvline(-3, color='red',    ls='--', lw=1.2, label='-3σ')
ax2.axvline(-5, color='orange', ls='--', lw=1.2, label='-5σ')
ax2.set_xlabel('Pre-pulse min  (× noise_std)')
ax2.set_ylabel('count')
ax2.set_title('Pre-pulse dip distribution')
ax2.legend()
plt.tight_layout()
plt.show()

print(f"总计: {len(snr_list)} 个 event-channel 对")
print(f"SNR   min={min(snr_list):.1f}  median={np.median(snr_list):.1f}  max={max(snr_list):.1f}")
print(f"Dip   min={min(glitch_depth_list):.2f}  median={np.median(glitch_depth_list):.2f}  p5={np.percentile(glitch_depth_list,5):.2f}")

In [ ]:
# ── 诊断：SNR 分布 + pre-pulse dip 分布，用于确定筛选倍数 ──────────────────────
fs  = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
N   = 5000   # quiet region 样本数

snr_list         = []   # 每个 event 的 SNR
glitch_depth_list = []  # pre-pulse dip 深度（单位：noise_std 倍数）

for Z_choice in ['Z1', 'Z2', 'Z3']:
    for i_det in lst:
        x0_ms = np.arange(len(events[0][Z_choice][i_det])) / fs * 1000

        for i in range(len(events)):
            y_norm, noise_std_norm, ok = preprocess_event(events[i][Z_choice][i_det].astype(np.int32))
            if not ok:
                continue

            rise_ms, rise_idx = find_rise(y_norm, x0_ms)
            if rise_idx is None:
                continue

            # 自动选 quiet region：事件早 → 取末尾，事件晚 → 取前段
            if rise_idx > N + 500:
                noise_std = np.std(y_norm[:N])
            else:
                noise_std = np.std(y_norm[-N:])

            if noise_std <= 0:
                continue

            # SNR：归一化后 peak=1，所以 SNR = 1 / noise_std
            snr_list.append(1.0 / noise_std)

            # pre-pulse dip（对齐后）
            x_aligned = align_event(y_norm, x0_ms)
            if x_aligned is None:
                continue
            mask = (x_aligned >= 24.9) & (x_aligned <= 25.25)
            if np.any(mask):
                depth = float(np.min(y_norm[mask])) / noise_std
                glitch_depth_list.append(depth)

# ── 画图 ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(snr_list, bins=60, color='steelblue', edgecolor='white')
ax1.set_xlabel('SNR  (peak / noise_std)')
ax1.set_ylabel('count')
ax1.set_title('Good event SNR distribution\n→ 看 signal 和 noise 之间有没有 gap')

ax2.hist(glitch_depth_list, bins=60, color='steelblue', edgecolor='white')
ax2.axvline(-3, color='red',    ls='--', lw=1.2, label='-3σ')
ax2.axvline(-5, color='orange', ls='--', lw=1.2, label='-5σ')
ax2.set_xlabel('Pre-pulse min  (× noise_std)')
ax2.set_ylabel('count')
ax2.set_title('Pre-pulse dip distribution\n→ glitch 事件应在左侧明显分离')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"总计: {len(snr_list)} 个 event-channel 对")
print(f"SNR  —— min: {min(snr_list):.1f},  median: {np.median(snr_list):.1f},  max: {max(snr_list):.1f}")
print(f"Dip  —— min: {min(glitch_depth_list):.2f},  median: {np.median(glitch_depth_list):.2f},  p5: {np.percentile(glitch_depth_list, 5):.2f}")


In [ ]:
# ── 诊断：SNR 分布 + pre-pulse dip 分布，用于确定筛选倍数 ──────────────────
fs  = 625000
lst = ['PFS1','PCS1','PDS1','PBS1','PES1','PFS2','PCS2','PBS2','PES2','PDS2','PAS2']
N   = 5000

snr_list          = []
glitch_depth_list = []

for Z_choice in ['Z1', 'Z2', 'Z3']:
    for i_det in lst:
        x0_ms = np.arange(len(events[0][Z_choice][i_det])) / fs * 1000
        for i in range(len(events)):
            y_norm, noise_std_norm, ok = preprocess_event(events[i][Z_choice][i_det].astype(np.int32))
            if not ok:
                continue
            rise_ms, rise_idx = find_rise(y_norm, x0_ms)
            if rise_idx is None:
                continue
            noise_std = np.std(y_norm[:N]) if rise_idx > N + 500 else np.std(y_norm[-N:])
            if noise_std <= 0:
                continue
            snr_list.append(1.0 / noise_std)
            x_aligned = align_event(y_norm, x0_ms)
            if x_aligned is None:
                continue
            mask = (x_aligned >= 24.9) & (x_aligned <= 25.25)
            if np.any(mask):
                glitch_depth_list.append(float(np.min(y_norm[mask])) / noise_std)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(snr_list, bins=60, color='steelblue', edgecolor='white')
ax1.set_xlabel('SNR  (peak / noise_std)')
ax1.set_ylabel('count')
ax1.set_title('Good event SNR distribution')

ax2.hist(glitch_depth_list, bins=60, color='steelblue', edgecolor='white')
ax2.axvline(-3, color='red',    ls='--', lw=1.2, label='-3σ')
ax2.axvline(-5, color='orange', ls='--', lw=1.2, label='-5σ')
ax2.set_xlabel('Pre-pulse min  (× noise_std)')
ax2.set_ylabel('count')
ax2.set_title('Pre-pulse dip distribution')
ax2.legend()
plt.tight_layout()
plt.show()

print(f'总计: {len(snr_list)} 个 event-channel 对')
print(f'SNR   min={min(snr_list):.1f}  median={np.median(snr_list):.1f}  max={max(snr_list):.1f}')
print(f'Dip   min={min(glitch_depth_list):.2f}  median={np.median(glitch_depth_list):.2f}  p5={np.percentile(glitch_depth_list,5):.2f}')

In [ ]:
总计: 26096 个 event-channel 对
SNR   min=15.9  median=130.9  max=1481.6
Dip   min=-49.53  median=-2.34  p5=-3.37